## **EVALUACIÓN ZERO-SHOT**

En este fichero se evalúa el rendimiento de todos los modelos de SAM estudiados en el proyecto utilizando el conjunto de datos PASCAL-S.

En este primer bloque de código se importan las librerías necesarias para la ejecución del fichero completo. Además, se importan las funciones de métricas y de medición de inferencia. Para ello se tuvo que añadir la raíz del proyecto al path de Python para que estos imports funcionen correctamente. Finalmente, se define la ruta al datset con el que entrenaremos los modelos.

In [2]:
from ultralytics import SAM
from ultralytics.models.sam import SAM3SemanticPredictor
import numpy as np
import os
import cv2
import pandas as pd
import time
import sys
import os


root_path = os.path.abspath('..')
if root_path not in sys.path:
    sys.path.append(root_path)

from utils.segmentation_quality_metrics import boundary_iou, hausdorff_95, compute_all_metrics
from utils.efficiency_metrics import measure_inference_central_point, measure_inference_sam3_prompt_zero_shot

dataset = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\PASCAL-S\\Test"


La función **evaluate_model** es idéntica para los modelos: SAM 1 Base, SAM 1 Large, SAM 2 Base, SAM 2 Large, SAM 2.1 Base, SAM 2.1 Large y SAM 3. Para cada imagen, se carga su máscara y se calcula el punto central a partir de los píxeles activos de la máscara, descartando las imágenes cuya máscara esté vacía. Cuando el modelo devuelve varias máscaras candidatas, se selecciona la que mayor IoU obtiene con el ground truth. La máscara predicha se redimensiona al tamaño original para hacerla comparable con el ground truth y se calculan todas las métricas. Finalmente, el tiempo de evaluación se añade al diccionario de resultados, que se guarda en un CSV acumulando los resultados de cada modelo.

En el caso de SAM 3, se evalúa además una variante adicional que usa un prompt de texto en lugar de un punto durante la inferencia. En lugar de instanciar el modelo con SAM, se usa SAM3SemanticPredictor con una configuración específica, y el prompt que se le pasa a la inferencia es "salient object" (objeto saliente), adecuado para el tipo de imágenes de PASCAL-S.

**SAM 1 BASE**

In [3]:
def evaluate_model(dataset):
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_b.pt")

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    images_dir = os.path.join(dataset, "Image")
    masks_dir = os.path.join(dataset, "GT")

    for img_file in os.listdir(images_dir):
        img_name = os.path.splitext(img_file)[0]
        img_path  = os.path.join(images_dir, img_file)
        mask_path = os.path.join(masks_dir, img_name + ".png")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        coords = np.argwhere(gt_mask)
        if len(coords) == 0:
            continue

        ymin, xmin = coords.min(axis=0)
        ymax, xmax = coords.max(axis=0)
        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        results, latency, vram = measure_inference_central_point(model, img_path, central_point)

        if results[0].masks is None or len(results[0].masks.data) == 0:
            continue

        best_idx  = 0
        masks = results[0].masks.data.cpu().numpy()
        H, W      = gt_mask.shape

        if len(masks) > 1:
            ious = []
            for m in masks:
                m_resized = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)
                intersection = np.logical_and(m_resized, gt_mask).sum()
                union = np.logical_or(m_resized, gt_mask).sum()
                ious.append(intersection / union if union > 0 else 0)
            best_idx = np.argmax(ious)

        pred_mask = masks[best_idx]
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam_b"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results

start_time = time.time()
results = evaluate_model(dataset)
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_pascals.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


**SAM 1 LARGE**

In [4]:
def evaluate_model(dataset):
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_l.pt")

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    images_dir = os.path.join(dataset, "Image")
    masks_dir = os.path.join(dataset, "GT")

    for img_file in os.listdir(images_dir):
        img_name = os.path.splitext(img_file)[0]
        img_path  = os.path.join(images_dir, img_file)
        mask_path = os.path.join(masks_dir, img_name + ".png")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        coords = np.argwhere(gt_mask)
        if len(coords) == 0:
            continue

        ymin, xmin = coords.min(axis=0)
        ymax, xmax = coords.max(axis=0)
        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        results, latency, vram = measure_inference_central_point(model, img_path, central_point)

        if results[0].masks is None or len(results[0].masks.data) == 0:
            continue

        best_idx  = 0
        masks = results[0].masks.data.cpu().numpy()
        H, W      = gt_mask.shape

        if len(masks) > 1:
            ious = []
            for m in masks:
                m_resized = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)
                intersection = np.logical_and(m_resized, gt_mask).sum()
                union = np.logical_or(m_resized, gt_mask).sum()
                ious.append(intersection / union if union > 0 else 0)
            best_idx = np.argmax(ious)

        pred_mask = masks[best_idx]
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam_l"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results

start_time = time.time()
results = evaluate_model(dataset)
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_pascals.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


**SAM 2 BASE**

In [5]:
def evaluate_model(dataset):
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2_b.pt")

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    images_dir = os.path.join(dataset, "Image")
    masks_dir = os.path.join(dataset, "GT")

    for img_file in os.listdir(images_dir):
        img_name = os.path.splitext(img_file)[0]
        img_path  = os.path.join(images_dir, img_file)
        mask_path = os.path.join(masks_dir, img_name + ".png")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        coords = np.argwhere(gt_mask)
        if len(coords) == 0:
            continue

        ymin, xmin = coords.min(axis=0)
        ymax, xmax = coords.max(axis=0)
        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        results, latency, vram = measure_inference_central_point(model, img_path, central_point)

        if results[0].masks is None or len(results[0].masks.data) == 0:
            continue

        best_idx  = 0
        masks = results[0].masks.data.cpu().numpy()
        H, W      = gt_mask.shape

        if len(masks) > 1:
            ious = []
            for m in masks:
                m_resized = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)
                intersection = np.logical_and(m_resized, gt_mask).sum()
                union = np.logical_or(m_resized, gt_mask).sum()
                ious.append(intersection / union if union > 0 else 0)
            best_idx = np.argmax(ious)

        pred_mask = masks[best_idx]
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam2_b"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results

start_time = time.time()
results = evaluate_model(dataset)
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_pascals.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


**SAM 2 LARGE**

In [6]:
def evaluate_model(dataset):
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2_l.pt")

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    images_dir = os.path.join(dataset, "Image")
    masks_dir = os.path.join(dataset, "GT")

    for img_file in os.listdir(images_dir):
        img_name = os.path.splitext(img_file)[0]
        img_path  = os.path.join(images_dir, img_file)
        mask_path = os.path.join(masks_dir, img_name + ".png")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        coords = np.argwhere(gt_mask)
        if len(coords) == 0:
            continue

        ymin, xmin = coords.min(axis=0)
        ymax, xmax = coords.max(axis=0)
        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        results, latency, vram = measure_inference_central_point(model, img_path, central_point)

        if results[0].masks is None or len(results[0].masks.data) == 0:
            continue

        best_idx  = 0
        masks = results[0].masks.data.cpu().numpy()
        H, W      = gt_mask.shape

        if len(masks) > 1:
            ious = []
            for m in masks:
                m_resized = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)
                intersection = np.logical_and(m_resized, gt_mask).sum()
                union = np.logical_or(m_resized, gt_mask).sum()
                ious.append(intersection / union if union > 0 else 0)
            best_idx = np.argmax(ious)

        pred_mask = masks[best_idx]
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam2_l"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results

start_time = time.time()
results = evaluate_model(dataset)
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_pascals.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


**SAM 2.1 BASE**

In [7]:
def evaluate_model(dataset):
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2.1_b.pt")

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    images_dir = os.path.join(dataset, "Image")
    masks_dir = os.path.join(dataset, "GT")

    for img_file in os.listdir(images_dir):
        img_name = os.path.splitext(img_file)[0]
        img_path  = os.path.join(images_dir, img_file)
        mask_path = os.path.join(masks_dir, img_name + ".png")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        coords = np.argwhere(gt_mask)
        if len(coords) == 0:
            continue

        ymin, xmin = coords.min(axis=0)
        ymax, xmax = coords.max(axis=0)
        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        results, latency, vram = measure_inference_central_point(model, img_path, central_point)

        if results[0].masks is None or len(results[0].masks.data) == 0:
            continue

        best_idx  = 0
        masks = results[0].masks.data.cpu().numpy()
        H, W      = gt_mask.shape

        if len(masks) > 1:
            ious = []
            for m in masks:
                m_resized = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)
                intersection = np.logical_and(m_resized, gt_mask).sum()
                union = np.logical_or(m_resized, gt_mask).sum()
                ious.append(intersection / union if union > 0 else 0)
            best_idx = np.argmax(ious)

        pred_mask = masks[best_idx]
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam2.1_b"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results

start_time = time.time()
results = evaluate_model(dataset)
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_pascals.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


**SAM 2.1 LARGE**

In [8]:
def evaluate_model(dataset):
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2.1_l.pt")

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    images_dir = os.path.join(dataset, "Image")
    masks_dir = os.path.join(dataset, "GT")

    for img_file in os.listdir(images_dir):
        img_name = os.path.splitext(img_file)[0]
        img_path  = os.path.join(images_dir, img_file)
        mask_path = os.path.join(masks_dir, img_name + ".png")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        coords = np.argwhere(gt_mask)
        if len(coords) == 0:
            continue

        ymin, xmin = coords.min(axis=0)
        ymax, xmax = coords.max(axis=0)
        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        results, latency, vram = measure_inference_central_point(model, img_path, central_point)

        if results[0].masks is None or len(results[0].masks.data) == 0:
            continue

        best_idx  = 0
        masks = results[0].masks.data.cpu().numpy()
        H, W      = gt_mask.shape

        if len(masks) > 1:
            ious = []
            for m in masks:
                m_resized = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)
                intersection = np.logical_and(m_resized, gt_mask).sum()
                union = np.logical_or(m_resized, gt_mask).sum()
                ious.append(intersection / union if union > 0 else 0)
            best_idx = np.argmax(ious)

        pred_mask = masks[best_idx]
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam2.1_l"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results

start_time = time.time()
results = evaluate_model(dataset)
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_pascals.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


**SAM 3**

In [9]:
def evaluate_model(dataset):
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam3.pt")

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    images_dir = os.path.join(dataset, "Image")
    masks_dir = os.path.join(dataset, "GT")

    for img_file in os.listdir(images_dir):
        img_name = os.path.splitext(img_file)[0]
        img_path  = os.path.join(images_dir, img_file)
        mask_path = os.path.join(masks_dir, img_name + ".png")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        coords = np.argwhere(gt_mask)
        if len(coords) == 0:
            continue

        ymin, xmin = coords.min(axis=0)
        ymax, xmax = coords.max(axis=0)
        w = xmax - xmin
        h = ymax - ymin

        central_point = [[xmin + w / 2, ymin + h / 2]]

        results, latency, vram = measure_inference_central_point(model, img_path, central_point)

        if results[0].masks is None or len(results[0].masks.data) == 0:
            continue

        best_idx  = 0
        masks = results[0].masks.data.cpu().numpy()
        H, W      = gt_mask.shape

        if len(masks) > 1:
            ious = []
            for m in masks:
                m_resized = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)
                intersection = np.logical_and(m_resized, gt_mask).sum()
                union = np.logical_or(m_resized, gt_mask).sum()
                ious.append(intersection / union if union > 0 else 0)
            best_idx = np.argmax(ious)

        pred_mask = masks[best_idx]
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam3"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results

start_time = time.time()
results = evaluate_model(dataset)
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_pascals.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


c:\Users\DanielTalavera\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post1)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
c:\Users\DanielTalavera\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must

**SAM 3 USANDO PROMPTS  DE TEXTO PARA SEGMENTAR**

In [10]:
def evaluate_model(dataset, text_prompt="salient object"):
    overrides = dict(
        conf=0.25,
        task="segment",
        mode="predict",
        model="C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam3.pt",
        device="cuda",
    )

    predictor = SAM3SemanticPredictor(overrides=overrides)

    iou_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    dice_scores = []
    specificity_scores = []
    f2_scores = []
    pixel_accuracy_scores = []
    boundary_iou_scores = []
    hausdorff_95_scores = []
    latency_scores = []
    vram_scores = []

    images_dir = os.path.join(dataset, "Image")
    masks_dir = os.path.join(dataset, "GT")

    for img_file in os.listdir(images_dir):
        img_name = os.path.splitext(img_file)[0]
        img_path  = os.path.join(images_dir, img_file)
        mask_path = os.path.join(masks_dir, img_name + ".png")

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = np.squeeze(gt_mask)
        gt_mask = (gt_mask > 127).astype(bool)

        if gt_mask.sum() == 0:
            continue

        results, latency, vram = measure_inference_sam3_prompt_zero_shot(predictor, img_path, text_prompt)

        if results[0].masks is None or len(results[0].masks.data) == 0:
            continue

        best_idx  = 0
        masks = results[0].masks.data.cpu().numpy()
        H, W      = gt_mask.shape

        if len(masks) > 1:
            ious = []
            for m in masks:
                m_resized = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)
                intersection = np.logical_and(m_resized, gt_mask).sum()
                union = np.logical_or(m_resized, gt_mask).sum()
                ious.append(intersection / union if union > 0 else 0)
            best_idx = np.argmax(ious)

        pred_mask = masks[best_idx]
        pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

        iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
        iou_scores.append(iou)
        precision_scores.append(precision)
        f1_scores.append(f1)
        recall_scores.append(recall)
        dice_scores.append(dice)
        specificity_scores.append(specificity)
        f2_scores.append(f2)
        pixel_accuracy_scores.append(pixel_accuracy)
        boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
        hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
        latency_scores.append(latency)
        vram_scores.append(vram)

    mean_iou = np.mean(iou_scores)
    mean_f1 = np.mean(f1_scores)
    mean_recall = np.mean(recall_scores)
    mean_precision = np.mean(precision_scores)
    mean_dice = np.mean(dice_scores)
    mean_specificity = np.mean(specificity_scores)
    mean_f2 = np.mean(f2_scores)
    mean_pixel_accuracy = np.mean(pixel_accuracy_scores)
    mean_boundary_iou = np.mean(boundary_iou_scores)
    mean_hausdorff_95 = np.mean(hausdorff_95_scores)
    mean_latency = np.mean(latency_scores)
    mean_vram = np.mean(vram_scores)

    results = {
         "modelo": ["sam3_prompt"],
         "mean_iou": [mean_iou],
         "mean_f1": [mean_f1],
         "mean_recall": [mean_recall],
         "mean_precision": [mean_precision],
         "mean_dice": [mean_dice],
         "mean_specificity": [mean_specificity],
         "mean_f2": [mean_f2],
         "mean_pixel_accuracy": [mean_pixel_accuracy],
         "mean_boundary_iou": [mean_boundary_iou],
         "mean_hausdorff_95": [mean_hausdorff_95],
         "mean_latency_ms": [mean_latency],
         "mean_vram_mb": [mean_vram]
    }

    return results

start_time = time.time()
results = evaluate_model(dataset)
eval_time = time.time() - start_time

results["eval_time_minutes"] = [round(eval_time / 60, 2)]

df = pd.DataFrame(results)
output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_pascals.csv"

if os.path.exists(output_path):
    df.to_csv(output_path, mode='a', header=False, index=False)
else:
    df.to_csv(output_path, index=False)


Ultralytics 8.4.26  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\PASCAL-S\Test\Image\1.jpg: 644x644 (no detections), 170.8ms
Speed: 4.2ms preprocess, 170.8ms inference, 1.1ms postprocess per image at shape (1, 3, 644, 644)
Results saved to C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\SAM-zero-shot\runs\segment\predict31
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\PASCAL-S\Test\Image\10.jpg: 644x644 (no detections), 75.1ms
Speed: 2.9ms preprocess, 75.1ms inference, 1.7ms postprocess per image at shape (1, 3, 644, 644)
Results saved to C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\SAM-zero-shot\runs\segment\predict31
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 C:\Users\DanielTa